# 11 — Agentic AI Orchestration for the AI Commerce Analytics Platform

This notebook explains and exercises the implementation under `backend/app/agents`, `backend/app/tools`, and `backend/app/guardrails`. It calls the same backend `run_agent` graph used by FastAPI; it does not create a notebook-only agent or retrain existing models.

**Implemented scope:** one LangGraph supervisor over existing deterministic services, request-scoped state, bounded retries, actual churn/CLV fallbacks, basic injection and PII checks, evidence-based synthesis, and a result-aware observe → decide → act loop. **Not implemented:** autonomous specialist agents, long-term memory, toxicity classification, or misuse classification. Contact with Ollama is optional; deterministic routing remains available.

## 1. Existing platform context

```text
Existing platform: processed commerce data → RFM → saved churn/CLV → forecast/recommendations/sentiment → RAG/FAISS
Added layer: one LangGraph supervisor selects among those existing services and reviews each returned result
```

The models and services remain deterministic tools. The LLM can classify a request and choose an eligible next tool; it does not run model inference itself.

## 2. Normal RAG vs agentic orchestration

```text
RAG: question → retrieve documents → grounded evidence
Agent: question → understand intent → select one eligible tool → observe/validate result → decide next action → synthesize
```

RAG is sufficient for indexed knowledge questions. Structured metrics, saved predictions, cohort building, conditional investigations, and recommendations need the relevant service tools. The RAG retriever is one registered tool, not the destination for every question.

## 3. Resolve the project and import the production implementation

Run from the repository root or this `notebooks/` directory. The code resolves the repository by its backend marker instead of relying on a Windows-specific path.

In [ ]:
from pathlib import Path
import sys

def find_project_root(start: Path | None = None) -> Path:
    current=(start or Path.cwd()).resolve()
    for path in (current,*current.parents):
        if (path/"backend/app/agents/supervisor.py").exists():
            return path
    raise FileNotFoundError("AI-Commerce-Analytics-Platform root not found")

PROJECT_ROOT=find_project_root()
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0,str(PROJECT_ROOT))
from backend.app.agents.supervisor import agent_graph, run_agent, MAX_ITERATIONS
from backend.app.agents.state import AgentState
from backend.app.agents.planner import understand_request
from backend.app.agents.selector import select_next_action
from backend.app.tools.registry import TOOL_REGISTRY, validate_inputs, validate_output
from backend.app.evaluation.golden_dataset import GOLDEN_DATASET
print("Project:",PROJECT_ROOT)
print("Allowlisted tools:",list(TOOL_REGISTRY))

## 4. Actual project integration map

| Runtime concern | Implementation |
|---|---|
| State, planning, selection, graph | `backend/app/agents/state.py`, `planner.py`, `selector.py`, `supervisor.py` |
| Cohort and bounded service adapters | `backend/app/tools/` |
| Input injection/PII and output checks | `backend/app/guardrails/`, graph output guard |
| Runtime evaluation and goldens | `backend/app/evaluation/` |
| FastAPI | `POST /api/v1/agent/ask` in `backend/app/api/agent.py` |
| Streamlit | `streamlit/pages/13_Agentic_Analyst.py` |
| Docker | LangGraph dependency in `backend/requirements.txt`, planner config in Compose |

The organization stays under `backend/app` because that is where the existing API service layer lives. Duplicate agent implementations under `src/` are intentionally avoided.

## 5. Agent state

The typed state in `backend/app/agents/state.py` carries the query, objectives, current tool, validated call/results, observations, next action, retry/fallback counts, memory context, guardrail result, anomalies, sources, final response, evaluation, and safe trace. Results are available to subsequent decisions through `tool_results` and `observations`; the runtime does not execute a prebuilt queue.

In [ ]:
print(AgentState.__annotations__.keys())
print("Graph nodes:",list(agent_graph.get_graph().nodes))
print("Maximum tool attempts/iterations:",MAX_ITERATIONS)

## 6. Registry contracts, input validation, and output validation

Each `ToolSpec` has a registered name, description, JSON-like schema, handler, output contract, retry limit, optional fallback tool, and risk level. Runtime validation checks required and extra inputs, types, ranges, enums, business constraints, output fields, and selected domain rules. Fallback tools are registered but marked non-selectable so the LLM cannot invoke them as ordinary goals.

In [ ]:
for name,spec in TOOL_REGISTRY.items():
    print(name,{"description":spec.description,"schema":spec.schema,"max_retries":spec.max_retries,
                "fallback_tool":spec.fallback_tool,"risk_level":spec.risk_level,"selectable":spec.selectable})

## 7. Planning is request understanding, not a task queue

`understand_request` returns an intent and allowed objectives. It does not ask the LLM for `[tool1, tool2, tool3]`. Invalid or unavailable local Ollama planning falls back to deterministic intent recognition. Obvious user requirements are merged back into the validated objective set so a planner cannot silently discard them.

In [ ]:
intent,objectives,planner_mode,llm_calls=understand_request(
    "Which high-value customers may churn and what could we recommend?",mode="rules")
print({"intent":intent,"objectives":objectives,"planner_mode":planner_mode,"llm_calls":llm_calls})

## 8. Result-aware tool selection

`select_next_action` uses the goals and actual validated results in state to build only feasible candidate actions. The LLM receives registry descriptions/schemas, eligible candidate names, and safe observation summaries; it never receives retrieved document bodies, raw customer feature rows, or tool arguments. The host constructs arguments from the query and results, then validates them. The deterministic selector is the fallback when Ollama fails.

## 9. LangGraph is the real observe → decide → act loop

The compiled graph routes one action at a time: select → execute → validate → observe → reason → select again or synthesize. Conditional edges handle retry, fallback, invalid output, blocked input, stop conditions, and the maximum iteration limit. The graph’s recursion limit is a separate secondary guard, not the agent iteration counter.

In [ ]:
print(agent_graph.get_graph().draw_mermaid())

## 10. Customer cohort and model feature contracts

`customer_cohort_tool` uses `src.features.customer.build_customer_features` over the existing master data. `high_value` uses historical monetary value at or above the 75th percentile; `at_risk` is descriptive recency >180 days. Cohort rows include the actual derived fields. Churn/CLV tools compare those columns with the saved estimator’s `feature_names_in_`; they never invent missing model inputs. A mismatch becomes a recorded tool failure.

In [ ]:
cohort_spec=TOOL_REGISTRY["customer_cohort_tool"]
print(cohort_spec.schema)
print("Churn estimator path:",PROJECT_ROOT/"models/customer_churn_model.joblib")
print("CLV estimator path:",PROJECT_ROOT/"models/customer_clv_model.joblib")

## 11. Execute a live query through the same backend graph

Set `RUN_LIVE_DEMOS=True` only after project dependencies, processed data, models, and any required FAISS index are available. Each query invokes the real `run_agent`; unavailable artifacts appear as errors/fallbacks. No demo result is fabricated.

In [ ]:
RUN_LIVE_DEMOS=False
if RUN_LIVE_DEMOS:
    revenue_result=run_agent("What is the total revenue?",planner_mode="rules",selector_mode="rules")
    display(revenue_result)
else:
    print("Live calls are disabled. Set RUN_LIVE_DEMOS=True to execute against this repository's services.")

## 12. Conditional multi-tool investigation

For “Which high-value customers are likely to churn and what products should we recommend?”, the deterministic path starts with a high-value cohort, then RFM. Once those actual customer rows exist, CLV and churn become eligible; churn receives the CLV-enriched rows if CLV succeeded. Recommendations use identifiers returned by previous results. If the CLV/churn tool fails, a registered fallback can run. The exact path can change when results, feature contracts, or the LLM selector decision differ.

In [ ]:
if RUN_LIVE_DEMOS:
    investigation=run_agent(
        "Which high-value customers are likely to churn and what products should we recommend?",
        planner_mode="rules",selector_mode="rules")
    display({key:investigation[key] for key in ("answer","tools_used","fallback_used","errors","evaluation","trace")})

## 13. Retry and real fallback

Transient failures receive at most two retries (per-tool limits also apply). Input/schema errors are not retried. After churn failure, the graph actually executes `rfm_risk_tool`; its output is explicitly marked as a recency heuristic and never presented as a trained churn probability. CLV failure can invoke `historical_value_tool`. RAG has no misleading string fallback: retrieval failure produces a controlled partial/unsupported answer.

In [ ]:
print("Churn fallback:",TOOL_REGISTRY["churn_prediction_tool"].fallback_tool)
print("CLV fallback:",TOOL_REGISTRY["clv_prediction_tool"].fallback_tool)
print("Fallback execution and retry assertions live in backend/tests/test_agent_runtime.py")

## 14. Iteration bounds and execution anomalies

Each tool attempt increments `iteration_count`; retries consume the same global budget. Reaching the limit stops the graph, records an anomaly, and preserves partial results. The runtime also detects repeated identical tool/result signatures, A/B tool oscillation, excessive retries/fallbacks, empty retrieval, and tool latency spikes. Business metric anomalies remain separate and use the project’s MAD-based detector.

## 15. Response synthesis and grounding

The graph has separate synthesis, output-guard, and evaluation nodes. The synthesizer turns successful typed results into a concise business summary; it does not JSON-dump the entire state and does not reveal chain-of-thought. The output guard redacts common email/phone patterns and checks retrieved-source provenance. `grounded` is a categorical evidence status, not a fabricated score.

## 16. Memory scope

Current memory is request-scoped LangGraph state and a compact list of completed tools. There is no SQLite, JSON, or other long-term user memory. Do not describe this implementation as cross-session memory.

## 17. Single-agent vs multi-agent

This implementation is **single-agent orchestration over deterministic tools**. `supervisor.py` is a graph/supervisor function, not a collection of autonomous customer, analytics, and knowledge agents. There is no multi-agent delegation.

## 18. Input and retrieval guardrails

Input rules enforce non-empty/length limits, block a few common prompt-injection patterns, block email/phone-like PII, and request clarification for a few obviously vague prompts. This is a demo detector, not a complete classifier. Retrieved documents are treated as untrusted data: the planner never sees retrieval output, and the selector sees only source/count summaries rather than document text. No toxicity or misuse classifier is installed.

## 19. Output safety and unsupported claims

Tool outputs pass contract and domain checks before entering observations. The response is assembled from validated results; model probabilities, CLV numbers, and recommendations must originate from their registered tools. If evidence is absent or retrieval sources fail validation, the response is marked unknown/false instead of inventing grounding. Common contact details are redacted; broader PII detection is not implemented.

## 20. Runtime evaluation

`backend/app/evaluation/agent_eval.py` executes requests and scores intent, first-tool selection, safe argument summaries, task completion, plan/tool order, retries, fallback rate/correctness where expected, tool calls, iterations, latency, failure, and grounding. The notebook golden set includes analytics, RFM, churn, CLV, forecast, recommendation, sentiment, RAG, conditional investigation, ambiguity, injection, PII, and loop-limit cases. Separate tests force invalid tool calls, retry/fallback, and empty RAG.

In [ ]:
from backend.app.evaluation.agent_eval import evaluate_runtime_cases,benchmark_agent_versions
print("Golden cases:",len(GOLDEN_DATASET))
print("Evaluation API:",evaluate_runtime_cases.__name__,benchmark_agent_versions.__name__)

## 21. Version benchmark

V1 uses deterministic request understanding and rules selection; V2 uses the schema-checked LLM planner with rules selection; V3 uses the LLM planner plus the dynamic result-aware selector. All execute the same golden requests and score actual tool sequence, argument validity, task completion, result, retries, fallbacks, guardrail, latency, and answer. V2/V3 only represent real LLM runs when Ollama is available; response modes show deterministic fallback when it is not.

In [ ]:
RUN_FULL_BENCHMARK=False
if RUN_FULL_BENCHMARK:
    versions=benchmark_agent_versions(run_agent)
    for version,result in versions.items():
        print(version,result["metrics"])
else:
    print("Benchmark disabled: it makes repeated real service/LLM calls. Enable when ready.")

## 22. Execution tests

The backend tests prove a prior CLV result is passed into churn selection, test execution order, retries and actual RFM fallback, schema/output rejection, guardrails, PII redaction, empty RAG, repeated calls, iteration limits, and benchmark-mode separation.

In [ ]:
# From the repository root, in the project environment after installing backend/requirements.txt:
# python -m pytest backend/tests/test_agent_runtime.py

## 23. Observability and safe trace

The API returns a safe execution trace with tool, status, redacted input summary, latency, attempt, fallback relationship, and sources. It does not return raw customer feature inputs. Logs omit full questions/feature rows. Prometheus tracks request outcome, tool calls/failures/latency, retries, fallbacks, iterations, LLM latency, and execution anomalies with low-cardinality labels. Token usage and cost are `None` because the current Ollama client path does not expose reliable accounting.

## 24. Streamlit integration

The **Agentic Analyst** page calls `POST /api/v1/agent/ask`, accepts a question, optional platform customer ID, and optional review text, then displays the summary, status, tool sequence, sources, errors/fallbacks, safe trace, and evaluation. It does not display hidden reasoning or raw tool arguments.

## 25. FastAPI integration

The API route is registered in `backend/app/main.py`. Request/response schemas live in `backend/app/schemas/contracts.py`; execution is synchronous in FastAPI’s worker pool. The existing project has no API authentication, so customer-scoped endpoints must not be exposed publicly without adding authentication and authorization.

## 26. Docker and local configuration

`backend/requirements.txt` includes LangGraph; `docker-compose.yml` passes `AGENT_PLANNER_ENABLED` and the planner timeout to the backend. Set `AGENT_PLANNER_ENABLED=false` for deterministic-only behavior. The existing Ollama service/host networking and project artifacts are reused; no new database or agent microservice is introduced.

## 27. Latency, cost, and tokens

End-to-end and tool latency are measured, and the current planner/selector LLM latency is counted when available. More decisions can add Ollama latency. Early finishing, bounded iterations, no unnecessary RAG, and result caching are possible optimizations. This implementation does not calculate token usage or currency cost; fields remain null until the runtime exposes reliable values.

## 28. KV cache

KV cache is an inference-engine optimization. The application can benefit from it but does not directly implement LLM internal KV-cache management.

## 29. End-to-end scenarios

With live calls enabled, try simple revenue analytics; high-risk cohort → RFM → churn; high-value cohort → RFM → CLV → churn → recommendations; business anomaly → conditional RAG investigation only if flags exist; RAG with source validation; and sentiment with provided review text. Failure/retry/fallback, injection, PII, max iterations, empty retrieval, and repeated execution are covered by tests.

## 30. Architecture that matches runtime

```mermaid
flowchart TD
 U[User] --> G[Input guard and PII / injection checks] --> P[Understand intent and objectives]
 P --> S[Result-aware tool selector]
 S --> E[Allowlisted tool executor]
 E --> V[Input/output contract validation]
 V -->|valid| O[Observe result and update request memory]
 V -->|retryable and budget remains| R[Bounded retry] --> E
 V -->|retries exhausted| F[Registered fallback tool] --> E
 V -->|invalid / no fallback / budget exhausted| Y[Synthesize partial evidence]
 O --> D[Reason: next safe action or done]
 D -->|continue| S
 D -->|done / repeated-call anomaly| Y
 Y --> OG[Output PII and source guard]
 OG --> Q[Runtime evaluation]
 Q --> M[Structured trace and Prometheus]
 M --> END[Grounded response]
 X[RAG result] -. untrusted document data; selector receives only source/count .-> O
```

There are no autonomous specialist-agent branches and no persistent memory node.

## 31. Interview-ready summary

I added a LangGraph-based, single-agent orchestration layer over the commerce platform’s existing ML and RAG services. It understands user intent, selects one schema-validated tool at a time, observes and validates results, and selects the next action using state and previous outputs. Retries are bounded; churn/CLV fallbacks execute real registered heuristic tools and are labeled as proxies. The API returns grounded evidence and safe telemetry; memory is request-scoped. This is not a multi-agent or long-term-memory system, and toxicity/misuse classification is not implemented.